In [2]:
import os
import numpy as np
import librosa
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score

In [3]:
def extract_mfcc(file_path, n_mfcc=40):
    try:
        audio, sample_rate = librosa.load(file_path, sr=None)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
        return np.mean(mfccs, axis=1)
    except Exception as e:
        print(f"Skipping {file_path}: {e}")
        return None

In [5]:
def load_folder(folder_path):
    rows, labels = [], []
    for label in ("real", "fake"):
        class_path = os.path.join(folder_path, label)
        for filename in os.listdir(class_path):
            if not filename.lower().endswith(".wav"):
                continue
            features = extract_mfcc(os.path.join(class_path, filename))
            if features is not None:
                rows.append(features)
                labels.append(label)
    return np.array(rows), np.array(labels)

In [6]:
BASE = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm"

print("Loading training data...")
X_train, y_train = load_folder(os.path.join(BASE, "training"))
print(f"Training: {X_train.shape}")

print("Loading test data...")
X_test, y_test = load_folder(os.path.join(BASE, "testing"))
print(f"Testing: {X_test.shape}")

Loading training data...


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1837
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1891
  warnings.warn(


Training: (53868, 40)
Loading test data...
Testing: (4634, 40)


In [8]:
print("Training Random Forest...")
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=67)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n--- Results ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"F1 (fake): {f1_score(y_test, y_pred, pos_label='fake'):.3f}")
print("\n", classification_report(y_test, y_pred))

Training Random Forest...

--- Results ---
Accuracy : 0.856
F1 (fake): 0.841

               precision    recall  f1-score   support

        fake       0.97      0.74      0.84      2370
        real       0.78      0.98      0.87      2264

    accuracy                           0.86      4634
   macro avg       0.88      0.86      0.86      4634
weighted avg       0.88      0.86      0.85      4634



In [10]:
importances = model.feature_importances_
top5 = np.argsort(importances)[-5:][::-1]
print("Top 5 most important MFCC coefficients:", top5)
print("Their importance scores:", importances[top5].round(4))

Top 5 most important MFCC coefficients: [12 16  0 25 23]
Their importance scores: [0.066  0.0558 0.0541 0.0476 0.0424]
